# 08 — Ba phương pháp mới, đo trọn trong MỘT lượt chạy

Ngân sách chỉ đủ chạy một lần, nên notebook này thiết kế để **một lượt GPU sinh ra cả họ
kết quả**: mọi mẫu sinh ra đều được lưu, và mọi biến thể tính lại được trên CPU ở `07`.

| phương pháp | cơ chế | tính lại trên CPU được không |
|---|---|---|
| **Self-consistency** | sinh K mẫu, bỏ phiếu theo giá trị THỰC THI | ✅ đường cong k = 1…K |
| **Ví dụ động (kNN)** | thay 2 ví dụ cố định bằng ví dụ truy hồi từ train | ⛔ phải chạy riêng một nấc |
| **Lượt sửa** | program bị executor từ chối → sinh lại kèm lý do lỗi | ✅ có `program_truoc_sua` |

Cả ba chỉ hỏi **executor**, không đụng đáp án vàng.

## Hai nấc, và vì sao đủ

```
08_tu_nhat_quan   engineered, ví dụ CỐ ĐỊNH, n=K, sửa-khi-lỗi
09_vidu_dong      engineered, ví dụ TRUY HỒI, n=K, sửa-khi-lỗi
```

Từ hai lượt đó, `07` rút ra được:

| phép so | lấy từ đâu | đo cái gì |
|---|---|---|
| `08` k=1 → k=K | trong cùng một nấc | **self-consistency** |
| `09` k=1 vs `08` k=1 | hai nấc, cùng k, cùng nhiệt độ | **ví dụ động** |
| `09` k=K vs `08` k=K | hai nấc, cùng k | ví dụ động **dưới** self-consistency (tương tác) |
| có/không `program_truoc_sua` | trong cùng nấc | **lượt sửa** |
| `08` k=1 vs nấc 2 | hai nấc, khác nhiệt độ | **nhiệt độ** (0,7 so với 0,1) |
| best-of-K | `cac_ea`/`cac_pa` | **trần** của self-consistency |

⚠ Nấc 2 chạy ở `temperature=0.1`, hai nấc này ở `TEMP_MOI`. Chênh lệch `09`k=K so với
nấc 2 vì thế gồm **cả** phần nhiệt độ — tách ra bằng phép so cuối bảng, đừng gộp.

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("05c_ace_basic_random_base", "Đối chứng — bullet ngẫu nhiên, prompt cơ bản"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("04_selfeval_base_moi", "Mục tiêu — self-eval + K mẫu + ví dụ truy hồi"),
    ("10_bo_chon",         "Mới — model tự chấm giữa các ứng viên"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.

# (1) Ô cài đặt có thật sự cài được không. Đọc metadata nên nhanh, không phải import.
#     Kiểm ở đây để lỗi pip lộ ra trong 1 giây, thay vì 20 phút nữa lúc nạp model.
from importlib.metadata import version as _ver, PackageNotFoundError as _NoPkg
_goi = {}
for _p in ("vllm", "unsloth", "transformers", "trl", "peft", "torch"):
    try:
        _goi[_p] = _ver(_p)
    except _NoPkg:
        _goi[_p] = None
print("[GÓI] " + " | ".join(f"{k}={v}" for k, v in _goi.items() if v))
# vLLM phải khớp CUDA của torch, nếu không unsloth CHẶN import dù gói vẫn có mặt —
# lúc đó cell nạp model báo "No module named 'vllm'" một cách khó hiểu.
# Wheel khớp CUDA có đuôi "+cuXXX" trong số phiên bản; bản PyPI thì không.
if _goi.get("vllm") and "+cu" not in _goi["vllm"]:
    print("[GÓI] ⚠ vllm=" + _goi["vllm"] + " là bản PyPI (dựng cho CUDA 13). Nếu cell nạp "
          "model báo \"No module named 'vllm'\" thì cài lại bằng wheel khớp CUDA — "
          "dòng WARNING của unsloth in sẵn URL đúng.")

_thieu = [k for k, v in _goi.items() if v is None]
if _thieu:
    raise RuntimeError(
        "Thiếu gói: " + ", ".join(_thieu) + " — ô cài đặt (cell #1) đã thất bại.\n\n"
        "Cách chữa: mở Cửa sổ dòng lệnh (góc dưới trái), chạy\n"
        "    pip install -U unsloth vllm\n"
        "xem lỗi thật, xong Restart session rồi chạy lại TỪ CELL #2 (bỏ qua cell #1).")

# vLLM 0.23 cấm toàn bộ transformers 5.x. Gói nào đó nâng lên 5 thì chặn ngay tại đây,
# đừng để phát hiện sau 4 phút nạp model. (sentence-transformers ≥ 6 là thủ phạm hay gặp.)
if str(_goi["transformers"]).split(".")[0] != "4":
    raise RuntimeError(
        "transformers=" + str(_goi["transformers"]) + " — vLLM 0.23 chỉ chạy với "
        "transformers 4.x, gói nào đó đã nâng nó lên.\n"
        "Chữa: pip install \"transformers==4.57.6\" rồi Restart session.")

# (2) Executor có tái tạo đúng nhãn vàng không.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1
# max_tokens thì KHÔNG giữ: nâng 3000 → 8192 vì ở mức cũ 5–10 % mẫu bị cắt giữa lúc
# suy nghĩ, mất trắng. Xem lý do đầy đủ ở ô cấu hình GPU.
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Runtime → Change runtime type → A100 GPU.")

# ═══ Tham số ẢNH HƯỞNG KẾT QUẢ — CỐ ĐỊNH trên mọi GPU ═══
# Trước đây max_seq đổi theo GPU (A100 15000 / L4 13500) nên hai máy cho kết quả
# không so thẳng được. Giờ khoá cứng: đổi GPU chỉ đổi tốc độ, không đổi đầu vào.
#
# TRẦN SINH = 4096. Đây là mức ĐO ĐƯỢC là tối ưu, không phải chọn bừa:
#     nấc 2, cùng prompt, cùng GPU, chỉ khác trần —
#       4096 → 30 lượt bị cắt | 28 mẫu mất | EA 0.6479 | 300 mẫu đúng
#       8192 → 30 lượt        | 28 mẫu     | EA 0.6479 | 300 mẫu đúng
#     Gấp đôi ngân sách cứu ĐÚNG 0 mẫu. Số mẫu vượt trần không phụ thuộc trần, nên 4096
#     đã qua điểm bão hoà; 8192 chỉ tốn thêm thời gian. (Dưới 4096 thì mất thêm mẫu.)
#
# ~6 % mẫu vẫn chạm trần — nay KHÔNG bỏ mặc nữa: run_pipeline vớt chúng bằng một lượt
# sinh lại với suy nghĩ TẮT (xem `vot_mau_bi_cat`). Đó mới là cách chữa, không phải trần.
#
# max_seq 17000 theo ngân sách (neo vào phép đo thật bằng tokenizer):
#     prompt bước 2 xấu nhất = 7464 + 4096 = 11560
#     ngân sách              = 17000 − 4096 = 12904   → dư 1344 token
# Ô §3 đo lại bằng tokenizer thật và tự cắt ngữ cảnh + báo động nếu tính sai.
#
# ⚠ ĐỪNG nâng tiếp. Đã có phép so SẠCH: nấc 2 chạy hai lần với CÙNG prompt engineered,
# cùng model, cùng GPU, chỉ khác trần token —
#     trần 4096 → 30 lượt sinh bị cắt | 28 mẫu mất trắng | EA 0.6479 | 300 mẫu đúng
#     trần 8192 → 30 lượt             | 28 mẫu           | EA 0.6479 | 300 mẫu đúng
# Gấp đôi ngân sách cứu được ĐÚNG 0 mẫu, đổi lại ~50 % thời gian (10,9 → 16,4 phút).
#
# Số mẫu vượt ngân sách KHÔNG phụ thuộc ngân sách → những lượt đó thực tế không có điểm
# dừng. Mà chúng cũng không lặp (§4 đo trung vị lặp = 0.0 ở nấc 2), nên repetition_penalty
# cũng không phải thuốc. Coi đây là sàn ~6 %, đều ở mọi nấc: ghi nhận rồi bỏ qua.
TEMPERATURE, MAX_TOKENS = 0.1, 4096
MAX_SEQ_LENGTH = 17000
REPETITION_PENALTY = 1.0

# ═══ Tham số chỉ ảnh hưởng TỐC ĐỘ — chỉnh theo VRAM ═══
if _VRAM < 20:
    raise RuntimeError(
        f"{_GPU} chỉ {_VRAM:.0f} GB — không đủ cho max_seq={MAX_SEQ_LENGTH}.\n"
        f"Hạ max_seq xuống thì kết quả KHÔNG so được với các nấc khác, nên thà dừng "
        f"còn hơn ra một con số không dùng được. Đổi sang L4 hoặc A100.")
# util giữ 0.85 (hạ từ 0.88 sau một lần vLLM không dựng nổi engine vì VRAM còn sót).
# MAX_NUM_SEQS trả về mức cũ được vì max_seq đã từ 25000 xuống 17000, áp lực KV giảm hẳn.
#
# BATCH_SIZE = 512 để 497 mẫu vào ĐÚNG MỘT LÔ. Đo từ log thật: lô 400 mẫu chạy
# 1,88 s/mẫu, lô 97 mẫu còn lại chạy 2,83 s/mẫu — chậm hơn 50 % vì không lấp đầy GPU mà
# vẫn phải đợi mẫu dài nhất. Gộp một lô tiết kiệm ~1,5 phút MỖI lượt sinh; nấc 4 và nấc 5
# có nhiều lượt nên cộng lại đáng kể.
elif _VRAM < 30:                           # L4 24GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.86, 16, 512
elif _VRAM < 60:                           # A100 40GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 48, 512
else:                                      # A100 80GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 128, 512
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"(cố định mọi GPU) | batch={BATCH_SIZE} max_num_seqs={MAX_NUM_SEQS} (theo VRAM)")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

# enable_prefix_caching: system prompt (~1 800 token) GIỐNG HỆT ở cả 497 request, nên
# vLLM chỉ cần prefill nó một lần rồi dùng lại. Tiết kiệm phần lớn thời gian prefill.
# Không đổi token sinh ra — mỗi request vẫn có seed riêng.
_NAP_KW = dict(model_name     = MODEL_NAME,
               dtype          = DTYPE,
               max_seq_length = MAX_SEQ_LENGTH,
               load_in_4bit   = True,
               fast_inference = True)
try:                                  # bản unsloth cũ không nhận tham số này
    import inspect as _insp
    if "enable_prefix_caching" in _insp.signature(
            FastLanguageModel.from_pretrained).parameters:
        _NAP_KW["enable_prefix_caching"] = True
except Exception:                                    # noqa: BLE001
    pass
NAP_AN_TOAN = False          # True = đã phải lùi về chế độ an toàn, có ghi vào meta

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **_NAP_KW, gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS)
except (RuntimeError, ValueError) as _e:
    # ── vLLM dựng engine hỏng vì CUDA ──
    # KHÔNG phải tải model hỏng: model đã nằm trên đĩa rồi. Lỗi ở bước cấp phát KV cache
    # và dựng CUDA graph — thường do VRAM trống ít hơn lần trước (GPU khác, hoặc tiến
    # trình cũ còn giữ bộ nhớ), khiến số block KV tính ra quá nhỏ.
    if "CUDA error" in str(_e) or "invalid argument" in str(_e):
        print("[MODEL] ⚠ vLLM KHÔNG dựng được engine (CUDA error).")
        print(f"[MODEL]   Đang dùng: max_seq={MAX_SEQ_LENGTH} util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}")
        try:
            _free, _tot = torch.cuda.mem_get_info()
            print(f"[MODEL]   VRAM trống: {_free/1024**3:.1f}/{_tot/1024**3:.1f} GB"
                  + ("   ← ĐÃ BỊ CHIẾM. Restart session rồi chạy lại TỪ Ô #2."
                     if _free / _tot < 0.9 else ""))
        except Exception:                                    # noqa: BLE001
            pass
        print("[MODEL]   Thử lại ở CHẾ ĐỘ AN TOÀN: bỏ CUDA graph, hạ VRAM và số chuỗi.")
        print("[MODEL]   Ba thứ đó chỉ đổi TỐC ĐỘ — mỗi request đã có seed riêng nên")
        print("[MODEL]   thành phần lô không ảnh hưởng token sinh ra.")
        gc.collect()
        torch.cuda.empty_cache()
        _an = dict(gpu_memory_utilization=min(GPU_MEM_UTIL, 0.80),
                   max_num_seqs=max(8, MAX_NUM_SEQS // 4))
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                **_NAP_KW, enforce_eager=True, **_an)
        except TypeError:                 # bản unsloth không nhận enforce_eager
            model, tokenizer = FastLanguageModel.from_pretrained(**_NAP_KW, **_an)
        GPU_MEM_UTIL = _an["gpu_memory_utilization"]
        MAX_NUM_SEQS = _an["max_num_seqs"]
        NAP_AN_TOAN = True
        print(f"[MODEL] ✅ nạp được ở chế độ an toàn (util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}) — chậm hơn, kết quả không đổi.")
    # ── Thiếu trọng số: shard safetensors tải dở còn trong cache ──
    elif "not initialized from checkpoint" in str(_e):
        raise RuntimeError(
            "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
            "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
            "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
            "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
            "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
            "    (`export` trong terminal không tới được kernel notebook.)\n\n"
            "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
            "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
            "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e
    else:
        raise

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

from collections import Counter as _Counter
LY_DO_DUNG = _Counter()          # finish_reason: "stop" = tự kết thúc, "length" = BỊ CẮT
LY_DO_THEO_BUOC = {}             # desc → Counter riêng, để tách bước 1 với bước 2


def ty_le_bi_cat():
    """Phần trăm lượt sinh bị cắt vì chạm max_tokens, TÍNH TỪ LẦN reset gần nhất."""
    t = sum(LY_DO_DUNG.values())
    return (LY_DO_DUNG.get("length", 0) / t) if t else 0.0


def bi_cat_theo_buoc():
    """Tỉ lệ bị cắt TÁCH RIÊNG cho bước 1 và bước 2.

    Phải tách vì prompt bước 2 (self-eval ở nấc 4, ACE ở nấc 5) chứa NGUYÊN lời giải
    bước 1, nên dài hơn bước 1 rất nhiều. Bước 2 bị cắt nhiều hơn nghĩa là phương pháp
    bị PHA LOÃNG — mất cơ hội sửa, chứ không phải sửa sai. Con số gộp chung không phân
    biệt được hai chuyện đó.

    Gom theo đuôi của desc ("vòng3/step1" và "vòng7/step1" cùng vào "step1").
    """
    gom = {}
    for k, c in LY_DO_THEO_BUOC.items():
        gom.setdefault(k.rsplit("/", 1)[-1], _Counter()).update(c)
    return {b: {"n": sum(c.values()), "bi_cat": c.get("length", 0),
                "ty_le": round(c.get("length", 0) / max(1, sum(c.values())), 4)}
            for b, c in sorted(gom.items())}


def in_bi_cat_theo_buoc():
    d = bi_cat_theo_buoc()
    if not d:
        return
    # In cả khi chỉ có MỘT bước: nấc 1 và 2 cũng cần biết tỉ lệ chạm trần của mình,
    # nếu không thì mãi tới nấc 4 mới thấy con số đó.
    print("   Bị cắt vì trần token, tách theo bước:")
    for b, v in d.items():
        print(f"     {b:<10}{v['ty_le']:>7.1%}  ({v['bi_cat']}/{v['n']} lượt)")
    if "step2" in d and "step1" in d and d["step2"]["ty_le"] > d["step1"]["ty_le"] + 0.02:
        print("     ⚠ bước 2 bị cắt nhiều hơn bước 1 → hiệu quả của phương pháp đang bị")
        print("       PHA LOÃNG (mất cơ hội sửa). Hiệu số đo được là cận DƯỚI.")


def dat_lai_bo_dem():
    """Gọi ngay trước mỗi nấc. Không gọi thì tỉ lệ là cộng dồn cả phiên — gồm cả
    lượt warmup và (ở nấc 5) toàn bộ pha A, không phản ánh nấc đang đo."""
    LY_DO_DUNG.clear()
    LY_DO_THEO_BUOC.clear()


def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        _res = model.fast_generate(chunk, **kw)
        for o in _res:                      # đếm lý do dừng để biết có bị cắt không
            for _x in o.outputs:            # sinh nhiều mẫu thì đếm CẢ K mẫu
                _r = getattr(_x, "finish_reason", "?")
                LY_DO_DUNG[_r] += 1
                if desc:
                    LY_DO_THEO_BUOC.setdefault(desc, _Counter())[_r] += 1
        # 1 mẫu → trả chuỗi (y như cũ); nhiều mẫu → trả list[str] cho self-consistency.
        outs.extend((o.outputs[0].text if len(o.outputs) == 1
                     else [_x.text for _x in o.outputs]) for o in _res)
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        _b = LY_DO_THEO_BUOC.get(desc, _Counter())
        _c, _n = _b.get("length", 0), max(1, sum(_b.values()))
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s"
              f" | bị cắt vì trần token: {_c/_n:.1%} ({_c} lượt)" + " "*8)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(tokenizer=tokenizer, model_name=MODEL_NAME)

# Nấc 2 — prompt hoàn chỉnh. Chỉ còn MỘT nền: ba chỗ bản gốc nói sai so với dữ liệu
# gold (table_* đọc theo cột, chuỗi #N, dấu câu "giảm") đã sửa sẵn trong PromptKit.
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False
_sys_dang_dung = prompt_kit.ENGINEERED_SYSTEM_PROMPT
print(f"[PROMPT] nền engineered | system prompt {len(_sys_dang_dung)} ký tự "
      f"(nấc 1 dùng {len(prompt_kit.BASIC_SYSTEM_PROMPT)})")

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         thang lồng nhau: basic={len(prompt_kit.BASIC_SYSTEM_PROMPT)} ký tự"
      f" ⊂ no_fewshot={len(prompt_kit.NO_FEWSHOT_SYSTEM_PROMPT)}"
      f" ⊂ engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)}"
      f" | self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

In [ ]:
# ═══════════ CẤU HÌNH PHƯƠNG PHÁP MỚI ═══════════
# K mẫu cho self-consistency. Chi phí tuyến tính theo K, nhưng đường cong k=1…K lấy
# được MIỄN PHÍ từ cùng một lượt chạy — nên đừng đặt K nhỏ rồi phải chạy lại.
#   K=5, 497 mẫu ≈ 80 phút/nấc trên A100.
SO_MAU = 5

# temperature 0.1 gần như tất định → K mẫu sẽ giống hệt nhau và bỏ phiếu vô nghĩa.
# Self-consistency CẦN đa dạng. 0.7 là mức chuẩn trong tài liệu gốc của phương pháp.
TEMP_MOI = 0.7
TOP_P_MOI = 0.95

# Số ví dụ truy hồi thay cho 2 ví dụ cố định. Đo trên bộ này: láng giềng gần nhất có
# cùng dãy phép với gold ở 46,5 % câu; trong top-3 thì 66,2 %.
SO_VI_DU = 3

SAMPLING_MOI = SamplingParams(temperature=TEMP_MOI, top_p=TOP_P_MOI,
                              max_tokens=MAX_TOKENS, n=SO_MAU,
                              repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
print(f"[MỚI] K={SO_MAU} mẫu | temp={TEMP_MOI} top_p={TOP_P_MOI} | "
      f"{SO_VI_DU} ví dụ truy hồi")
print(f"[MỚI] ước lượng: {SO_MAU}× lượt sinh ≈ {SO_MAU*16} phút/nấc")

# Kho ví dụ: dựng từ train, đã lọc mẫu có nhãn vàng nhiễu.
from vinumqa.fewshot import KhoViDu
_t0 = time.time()
KHO = KhoViDu(train_all)
print(f"[MỚI] kho ví dụ: {len(KHO)}/{len(train_all)} mẫu train "
      f"(đã bỏ nhãn nhiễu) — dựng trong {time.time()-_t0:.1f}s")

# Xem thử một khối ví dụ truy hồi được, để biết model sẽ nhìn thấy gì.
_s = test_all[0]
print(f"\n  CÂU TEST: {_s['qa']['question'][:90]}")
print(f"  GOLD    : {_s['qa']['program']}")
print("  ── ví dụ truy hồi ──")
print(KHO.van_ban_vi_du(_s["qa"]["question"], SO_VI_DU)[:700])

In [ ]:
# ═══════════ CỔNG KIỂM SỚM — 2 prompt, ~10 giây ═══════════
# Ba giả định dưới đây chưa từng được xác minh trên GPU. Sai bất kỳ cái nào thì cả
# nấc 80 phút ra rác. Kiểm bằng 2 prompt trước khi đốt thời gian thật.
_thu = [prompt_kit.step1(s, level="engineered",
                         vi_du_dong=KHO.van_ban_vi_du(s["qa"]["question"], SO_VI_DU))
        for s in test_all[:2]]
_ra = generate(_thu, SAMPLING_MOI, desc="cong-kiem")

# 1. vLLM có THẬT SỰ trả về K mẫu không?
assert isinstance(_ra[0], list), (
    f"generate() trả {type(_ra[0]).__name__}, không phải list — SamplingParams(n={SO_MAU}) "
    f"không có tác dụng. Self-consistency sẽ âm thầm thành K=1. Kiểm lại ô nạp model.")
assert len(_ra[0]) == SO_MAU, (
    f"xin {SO_MAU} mẫu nhưng nhận {len(_ra[0])}. Đừng chạy tiếp.")
print(f"[CỔNG] ✅ nhận đúng {len(_ra[0])} mẫu cho mỗi prompt")

# 2. K mẫu có KHÁC nhau không? temperature quá thấp thì bỏ phiếu vô nghĩa.
_prog = [dsl.extract_program_answer(x)[0] for x in _ra[0]]
_khac = len({p for p in _prog if p})
print(f"[CỔNG] {_khac}/{SO_MAU} chương trình khác nhau ở mẫu thử đầu: {_prog}")
if _khac <= 1:
    print(f"[CỔNG] ⚠ K mẫu GIỐNG HỆT nhau — bỏ phiếu sẽ không đổi được gì.")
    print(f"        temperature={TEMP_MOI} có thể còn thấp, hoặc câu này quá dễ.")
    print(f"        Xem mẫu thử thứ hai trước khi kết luận.")

# 3. Prompt có ví dụ ĐỘNG còn lọt ngân sách không? (ô trước chỉ đo ví dụ CỐ ĐỊNH)
_dai = [len(tokenizer(p).input_ids) for p in
        [prompt_kit.step1(s, level="engineered",
                          vi_du_dong=KHO.van_ban_vi_du(s["qa"]["question"], SO_VI_DU))
         for s in sorted(test_all, key=lambda x: -(len(str(x.get("table") or ""))))[:30]]]
_ns = MAX_SEQ_LENGTH - MAX_TOKENS
print(f"[CỔNG] prompt ví dụ động, 30 mẫu bảng dài nhất: max={max(_dai)} / ngân sách {_ns}")
assert max(_dai) <= _ns, (
    f"prompt ví dụ động tràn ngân sách ({max(_dai)} > {_ns}). Hạ SO_VI_DU xuống.")
print(f"[CỔNG] ✅ còn dư {_ns - max(_dai)} token")

## §A. Nấc `08_tu_nhat_quan` — self-consistency, ví dụ CỐ ĐỊNH

Giữ nguyên prompt của nấc 2, chỉ đổi hai thứ: sinh K mẫu thay vì 1, và bật lượt sửa.
Mọi mẫu đều được lưu nên `07` dựng được đường cong k = 1…K mà không chạy lại.

In [ ]:
dat_lai_bo_dem()
STAGE = "08_tu_nhat_quan"
print(f"\n{'═'*74}\n  NẤC: {STAGE} | K={SO_MAU} mẫu | ví dụ CỐ ĐỊNH | sửa-khi-lỗi"
      f" | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level="engineered", use_selfeval=False,
    sp_step1=SAMPLING_MOI, sp_step2=SAMPLING_MOI,
    sua_khi_loi=True, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics)
in_bi_cat_theo_buoc()
print(f"\n  Thời gian: {metrics['minutes']} phút")

In [ ]:
# ─── Đường cong self-consistency, tính NGAY tại đây (CPU, vài giây) ───
print(f"\n{'═'*74}\n  SELF-CONSISTENCY THEO k — nấc {STAGE}\n{'═'*74}")
print(f"{'k':>3}{'EA':>10}{'PA_strict':>12}{'Δ EA vs k=1':>14}")
_m1 = None
for _k in range(1, SO_MAU + 1):
    _mk = pipeline.summarize(pipeline.tu_nhat_quan(rows, test_all, _k), f"k={_k}")
    if _m1 is None:
        _m1 = _mk
    print(f"{_k:>3}{_mk['EA']:>10.4f}{_mk['PA_strict']:>12.4f}"
          f"{_mk['EA']-_m1['EA']:>+14.4f}")
_tran = pipeline.tran_best_of_k(rows)
print(f"\n  TRẦN best-of-{SO_MAU}: EA {_tran['EA_tran']:.4f} | PA {_tran['PA_tran']:.4f}")
print("  Trần là CẬN TRÊN nếu luôn chọn được mẫu đúng nhất — không phải kết quả đạt được.")
print("  Khoảng cách giữa k=K và trần cho biết còn bao nhiêu đất cho một bộ chọn tốt hơn.")

_n_sua = sum(1 for r in rows if r.get("da_sua"))
print(f"\n  Lượt sửa: {_n_sua} mẫu được sửa cho chạy được.")

In [ ]:
STAGE_DA_GHI = save_stage(STAGE, rows, metrics,
           extra={"prompt_level": "engineered", "self_eval": False,
                  "so_mau": SO_MAU, "temperature_moi": TEMP_MOI, "top_p": TOP_P_MOI,
                  "vi_du_dong": False, "sua_khi_loi": True,
                  "tran_best_of_k": _tran})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f}")

## §B. Nấc `09_vidu_dong` — thêm ví dụ truy hồi

Giống hệt §A, đổi đúng **một** thứ: 2 ví dụ cố định trong prompt bị thay bằng
`SO_VI_DU` ví dụ truy hồi từ train. Mọi phần khác của prompt giữ nguyên từng ký tự,
nên hiệu số `09` − `08` ở cùng k quy đúng về chuyện đổi ví dụ.

In [ ]:
dat_lai_bo_dem()
STAGE_B = "09_vidu_dong"
print(f"\n{'═'*74}\n  NẤC: {STAGE_B} | K={SO_MAU} mẫu | {SO_VI_DU} ví dụ TRUY HỒI"
      f" | sửa-khi-lỗi | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows_b = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level="engineered", use_selfeval=False,
    kho_vi_du=KHO, n_vi_du=SO_VI_DU,
    sp_step1=SAMPLING_MOI, sp_step2=SAMPLING_MOI,
    sua_khi_loi=True, desc=STAGE_B)
metrics_b = pipeline.summarize(rows_b, STAGE_B)
metrics_b["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics_b)
in_bi_cat_theo_buoc()
print(f"\n  Thời gian: {metrics_b['minutes']} phút")

In [ ]:
# ─── Hai nấc cạnh nhau, ở TỪNG mức k ───
print(f"\n{'═'*84}\n  VÍ DỤ ĐỘNG ĐÓNG GÓP BAO NHIÊU, Ở TỪNG MỨC k\n{'═'*84}")
print(f"{'k':>3}{'EA cố định':>13}{'EA truy hồi':>14}{'Δ EA':>9}"
      f"{'PA cố định':>13}{'PA truy hồi':>14}{'Δ PA':>9}")
for _k in range(1, SO_MAU + 1):
    _a = pipeline.summarize(pipeline.tu_nhat_quan(rows, test_all, _k), "a")
    _b = pipeline.summarize(pipeline.tu_nhat_quan(rows_b, test_all, _k), "b")
    print(f"{_k:>3}{_a['EA']:>13.4f}{_b['EA']:>14.4f}{_b['EA']-_a['EA']:>+9.4f}"
          f"{_a['PA_strict']:>13.4f}{_b['PA_strict']:>14.4f}"
          f"{_b['PA_strict']-_a['PA_strict']:>+9.4f}")

print(f"\n  TRẦN best-of-{SO_MAU} của nấc ví dụ động: "
      f"{pipeline.tran_best_of_k(rows_b)}")

# McNemar ở mức k đầy đủ — có ý nghĩa thống kê hay chỉ là nhiễu
_ka = pipeline.tu_nhat_quan(rows, test_all, SO_MAU)
_kb = pipeline.tu_nhat_quan(rows_b, test_all, SO_MAU)
for _key in ("ea", "pa_strict"):
    stats.compare_pair(_ka, _kb, key=_key,
                       label=f"ví dụ truy hồi so với cố định (k={SO_MAU})",
                       name_base="co_dinh", name_variant="truy_hoi")

In [ ]:
_tran_b = pipeline.tran_best_of_k(rows_b)
STAGE_DA_GHI_B = save_stage(STAGE_B, rows_b, metrics_b,
           extra={"prompt_level": "engineered", "self_eval": False,
                  "so_mau": SO_MAU, "temperature_moi": TEMP_MOI, "top_p": TOP_P_MOI,
                  "vi_du_dong": True, "so_vi_du": SO_VI_DU, "sua_khi_loi": True,
                  "tran_best_of_k": _tran_b})
print(f"\n  EA = {metrics_b['EA']:.4f} | PA_strict = {metrics_b['PA_strict']:.4f}")
print("  → Chạy 07_final_report.ipynb (CPU) để có bảng đầy đủ của cả hai nấc.")

## Đọc kết quả cho đúng

- **Đọc KTC, đừng so với một con số cố định.** Đo thật: chỉ đổi card mà nấc `03_sft`
  chênh 2,21 điểm EA. Ngưỡng 95 % là `1,96·√(b+c)/497`, tức ~3,7 điểm khi hai cấu hình
  bất đồng ở 90 mẫu. KTC chứa 0 = không phải phát hiện.
- **KTC chứa 0 = không kết luận được gì.** McNemar ở §B in sẵn kết luận đúng mực.
- **k=1 ở đây KHÔNG so thẳng được với nấc 2**: khác nhiệt độ (0,7 so với 0,1). Phép so
  đó đo *nhiệt độ*, và nó nằm ở `07`.
- Nếu đường cong k đi ngang từ k=2 → self-consistency không có đất trên bộ này; nói
  thẳng như vậy, đó vẫn là kết quả.
- Nếu **trần best-of-K cao hơn hẳn k=K** → phương pháp bỏ phiếu đang bỏ sót; chỗ đáng
  đầu tư tiếp là bộ chọn, không phải sinh thêm mẫu.